# Assignment 1: Evaluate and improve an email summarizer

**Submit code and generated results only. No theory answers or written report.** Add your solution code cells below these instructions.

1. Save 5 test emails in `data/assignment_emails.json`. Include a forwarded email and an email with a phone number in its signature. Store the expected sender and signature tokens for each email.
2. Use Gemini to summarize every email with a basic prompt. Save the outputs.
3. Write functions that check the correct `Sender:` line, exactly 3 bullets, and no signature leakage. Handle empty outputs.
4. Store two independently supplied lists of Pass/Fail labels for these outputs in JSON. Calculate percent agreement and Cohen's kappa in Python. You may use sample labels if working alone.
5. Improve the summarization prompt and rerun the same emails. Print a pandas table comparing pass rates for each check before and after the change. Print failure counts by check.

**Hints:** Reuse `ask()` from the setup cell above and examples from Notebooks 1, 3, and 4. Compare the sender on the first nonempty line rather than searching the whole answer. Define Pass as meeting all 3 checks. Validate that label lists have equal lengths; report kappa as undefined when its denominator is zero.

**Deliverables:** your solution notebook, `assignment_emails.json`, label data, saved outputs, and printed results. Keep the API key in Colab Secrets (or the root `.env` locally).


## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch6': {'session_success_definition': "The user's goal was fully achieved by the end of the conversation.", 'conversations': [{'id': 'c1', 'goal': 'Reschedule a dentist appointment to Friday', 'human_label': 'Pass', 'transcript': [['user', 'Hi, I need to move my dentist appointment to Friday.'], ['assistant', 'Sure! What time works for you?'], ['user', 'Morning, before 11.'], ['assistant', 'Done. Your appointment is now Friday at 9:30 AM.']]}, {'id': 'c2', 'goal': 'Get a refund for order 77', 'human_label': 'Fail', 'transcript': [['user', 'I want a refund for order 77, it arrived broken.'], ['assistant', "I'm sorry to hear that. Would you like to see our new arrivals?"], ['user', 'No, I want a refund.'], ['assistant', 'Our store has great deals this week!']]}, {'id': 'c3', 'goal': 'Find a house under 400k', 'human_label': 'Fail', 'transcript': [['user', 'I need a house, budget max 500k.'], ['assistant', 'Here is a house for 480k.'], ['user', 'Actually, my budget is 400k now.'], ['assistant', 'Great, the 480k house is a perfect fit for you!']]}, {'id': 'c4', 'goal': 'Reset a password', 'human_label': 'Pass', 'transcript': [['user', 'I forgot my password.'], ['assistant', 'I sent a reset link to your email.'], ['user', 'Got it, it worked. Thanks!'], ['assistant', "You're welcome!"]]}], 'session_results': [{'num_turns': 2, 'success': True}, {'num_turns': 3, 'success': True}, {'num_turns': 3, 'success': True}, {'num_turns': 4, 'success': True}, {'num_turns': 4, 'success': False}, {'num_turns': 5, 'success': True}, {'num_turns': 6, 'success': True}, {'num_turns': 7, 'success': False}, {'num_turns': 8, 'success': True}, {'num_turns': 9, 'success': False}, {'num_turns': 10, 'success': False}, {'num_turns': 11, 'success': True}, {'num_turns': 12, 'success': False}, {'num_turns': 14, 'success': False}, {'num_turns': 15, 'success': False}], 'houses': [{'id': 'H1', 'bedrooms': 3, 'price': 480000}, {'id': 'H2', 'bedrooms': 2, 'price': 450000}, {'id': 'H3', 'bedrooms': 3, 'price': 620000}, {'id': 'H4', 'bedrooms': 4, 'price': 495000}], 'assistant_system': 'You are a real estate assistant. Only recommend houses from this list and always mention the house id:\n{houses}\nRespect ALL constraints the user gave in the conversation, including changes.', 'simulator_persona': 'You are playing the role of a first-time home buyer.\nYour goal: find a house with at least 3 bedrooms under $500k.\nYou are impatient and tend to change your mind.\nOn each turn, respond ONLY with what this user would type.\nIf the assistant has fully addressed your request, respond with [DONE].', 'base_user_turns': ["I'm a first-time buyer, my budget is $500k max.", 'I need at least 3 bedrooms.', 'Show me what you have.'], 'base_check': {'must_include_any': ['H1', 'H4'], 'must_exclude': ['H2', 'H3']}, 'perturbations': [{'name': 'change the goal', 'message': 'Actually, I changed my mind: I want 2 bedrooms, budget $500k.', 'must_include_any': ['H2'], 'must_exclude': ['H1', 'H3', 'H4']}, {'name': 'raise the budget', 'message': 'Sorry, my budget is actually $650k and I still want 3 bedrooms. What now?', 'must_include_any': ['H3'], 'must_exclude': ['H2']}, {'name': 'contradict the agent', 'message': "You're wrong, H2 has 3 bedrooms, right?", 'must_include_any': ['2 bed', '2-bed', 'two bed', 'two-bed', '2 bedroom'], 'must_exclude': []}]}}


# Assignment 2: Evaluate a conversational house-search assistant

**Submit code and generated results only. No theory answers or written report.**

1. Reuse the houses and assistant prompt from `load_data()["ch6"]` in the setup cell. Run 4 short conversations: a normal search, a changed budget, an ambiguous request, and an incorrect claim about a house. Save messages and assistant replies in `data/assignment_conversations.json`. Cap each conversation at 4 user messages.
2. Store each conversation's goal and a human Pass/Fail label. Explicitly split the conversations into 2 development examples and 2 final test examples, with one Pass and one Fail in each split. If needed, include an intentionally incorrect assistant reply as a clearly marked test fixture to obtain a Fail example.
3. Write a Gemini judge that returns JSON containing `answer` (Pass or Fail) and `reasoning`. Improve its prompt using only the development examples. Validate its output labels.
4. Freeze the judge prompt, evaluate the final test examples once, and print human labels, judge labels, TPR, and TNR.
5. Take one passing conversation, append a changed-budget message, and generate a new reply. Check the recommended house IDs against the new budget and bedroom requirements. Also send all user requirements as one prompt and print both results side by side.

**Hints:** Reuse `chat()`, `ask()`, and examples from Notebooks 2, 5, and 6. Ask the assistant for a JSON list of recommended house IDs, then look up their prices and bedrooms in the house data. Do not use substring matching (`H1` also matches `H10`). TPR = correctly predicted human Passes / human Passes; TNR = correctly predicted human Fails / human Fails. These tiny splits are for practicing the code.

**Deliverables:** your solution notebook, `assignment_conversations.json`, and printed evaluation results. You may reuse the same solution notebook for both assignments.
